In [16]:
import pandas as pd 
import glob
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors
from tqdm import tqdm
from mordred import Calculator, descriptors
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV)
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               ExtraTreesClassifier, AdaBoostClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              matthews_corrcoef, confusion_matrix,
                              classification_report, precision_score,
                              recall_score, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

In [4]:
path ="C:/Users/angel/Desktop/Master/TFM/estructuras"
all_files = glob.glob(os.path.join(path, "*.csv"))

steps_log = []
df_list = []
li = []
for filename in all_files:
    df= pd.read_csv(filename, index_col=None, header=0, sep=';')
    li.append(df)
df_raw = pd.concat(li,axis=0, ignore_index=True)

#df_raw.to_csv('dataset_unificado.csv', index=False, sep= ';')

Aplicacion del criterio de clasificacion donde aquellos compuestos que necesitan menos de 1 µM serán considerados activos mientras que aquellos compuestos que necesitan mas o 1 µM serán considerados inactivos. 

In [5]:
def get_full_cleaned_dataset(df_raw):
    df = df_raw.copy()
    df['Standard Value'] = pd.to_numeric(df['Standard Value'], errors='coerce')
    df = df.dropna(subset= ['Standard Value','Standard Units', 'Smiles'])

    def convert_to_um(row):
        val = row['Standard Value']
        unit = str(row['Standard Units']).lower()
        smiles = row['Smiles']
        if unit =='nm':
            return val / 1000
        if 'um' in unit or 'µm' in unit:
            return val
        if 'ug.ml-1' in unit or 'ug/ml' in unit:
            mol = Chem.MolFromSmiles(smiles)
            if mol: 
                mw = Descriptors.MolWt(mol)
                return (val*1000) / mw
            return None
        return None
    df['IC50_uM'] = df.apply(convert_to_um, axis=1)
    df = df.dropna(subset=['IC50_uM'])

    df_train = df.groupby('Molecule ChEMBL ID').agg(
        smiles_std=('Smiles', 'first'),
        activity_uM=('IC50_uM', 'median'),
        n_measurements=('IC50_uM', 'count')
    ).reset_index()

    df_train['Label'] = (df_train['activity_uM'] < 1).astype(int)
    steps_log.append(('Deduplicate (median per molecule)', len(df), len(df_train)))
    
    return df_train
df_final = get_full_cleaned_dataset(df_raw)
df_final.to_csv('dataset_final_para_mordred.csv', sep=';', index=False)

Calculo de los descriptores fisicoquimicos a partir de los SMILES etiquetados anteriormente

In [6]:
mols, valid_idx = [], []

print("Convirtiendo SMILES a objetos moleculares...")
for i, row in df_final.iterrows():
    mol = Chem.MolFromSmiles(row["smiles_std"])
    if mol is not None:
        mols.append(mol)
        valid_idx.append(i)
df_mol = df_final.iloc[valid_idx].reset_index(drop=True)
print(f"Moléculas válidas para Mordred: {len(mols)}")

calc = Calculator(descriptors, ignore_3D=True)
print("Calculando descriptores 2D (esto puede tardar varios minutos)...")
df_desc = calc.pandas(mols)
print(f"Descriptores calculados inicialmente: {df_desc.shape[1]}")

Convirtiendo SMILES a objetos moleculares...
Moléculas válidas para Mordred: 537
Calculando descriptores 2D (esto puede tardar varios minutos)...


 37%|███▋      | 201/537 [00:14<00:30, 11.12it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 38%|███▊      | 203/537 [00:15<00:33,  9.83it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


...

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 50%|████▉     | 267/537 [00:20<00:12, 22.16it/s]

c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|██████████| 537/537 [00:33<00:00, 16.17it/s]


Descriptores calculados inicialmente: 1613


Limpieza de descriptores

In [7]:
# Columnas vacias
df_desc = df_desc.dropna(axis=1, how="all")
# Columnas con >20% NaN
thresh = int(len(df_desc) * 0.8)
df_desc = df_desc.dropna(axis=1, thresh=thresh)
# Solo columnas numericas
df_desc = df_desc.select_dtypes(include=[np.number])
# Imputar NaN restantes con mediana
imputer_init = SimpleImputer(strategy="median")
X_arr = imputer_init.fit_transform(df_desc)
X_full = pd.DataFrame(X_arr, columns=df_desc.columns)
# Varianza cero
vt = VarianceThreshold(threshold=0.0)
X_full = pd.DataFrame(
    vt.fit_transform(X_full),
    columns=X_full.columns[vt.get_support()]
)
y = df_mol["Label"].values
print(f"Descriptores tras limpieza: {X_full.shape[1]}")

X_out = X_full.copy()
X_out.insert(0, "ChEMBL_ID", df_mol["Molecule ChEMBL ID"].values)
X_out["Label"] = y
X_out.to_csv("paso4_descriptores_mordred.csv", index=False)
print(f"✅ paso4_descriptores_mordred.csv guardado.")

Descriptores tras limpieza: 944
✅ paso4_descriptores_mordred.csv guardado.


In [8]:
def correlation_filter(X: pd.DataFrame, threshold: float,
                        method: str = "pearson") -> list:
    corr_matrix = X.corr(method=method).abs()
    upper= corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
                             )
    to_drop = {col for col in upper.columns if any(upper[col] >= threshold)}
    retained = [col for col in X.columns if col not in to_drop]
    return retained
    
feature_sets = {}   # {"nombre": [lista_columnas_retenidas]}

CORR_THRESHOLDS   = [0.7, 0.8, 0.9]

for thr in CORR_THRESHOLDS:
    for method in ["pearson", "spearman"]:
        key = f"{method}_thr{thr}"
        retained = correlation_filter(X_full, thr, method)
        feature_sets[key] = retained
        print(f"  {key:30s} → {len(retained):4d} descriptores retenidos")

feature_sets["no_filter"] = list(X_full.columns)
print(f"  {'no_filter':30s} → {len(X_full.columns):4d} descriptores (sin filtro)")

pd.DataFrame(
    {"feature_set": list(feature_sets.keys()),
     "n_features":  [len(v) for v in feature_sets.values()]}
).to_csv("paso5_feature_sets.csv", index=False)
print(f"✅ paso5_feature_sets.csv guardado.", sep=';')

  pearson_thr0.7                 →  111 descriptores retenidos
  spearman_thr0.7                →  117 descriptores retenidos
  pearson_thr0.8                 →  192 descriptores retenidos
  spearman_thr0.8                →  184 descriptores retenidos
  pearson_thr0.9                 →  338 descriptores retenidos
  spearman_thr0.9                →  318 descriptores retenidos
  no_filter                      →  944 descriptores (sin filtro)
✅ paso5_feature_sets.csv guardado.


In [ ]:
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y,
    test_size=0.20,
    stratify=y,
    random_state= 42
)
print(f"Train: {len(y_train)} muestras | Test: {len(y_test)} muestras")
print(f"  Train activos: {y_train.sum()} ({y_train.mean():.1%})")
print(f"  Test  activos: {y_test.sum()} ({y_test.mean():.1%})")
 
# ── 6.2 Definición de modelos y rejillas de hiperparámetros ───────────────────
# Cada elemento: (nombre, estimador, param_grid)
# Los parámetros del pipeline se prefiján con 'clf__'
 
MODEL_DEFINITIONS = [
    # 1. Regresión Logística
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42),
        {
            "clf__C"      : [0.01, 0.1, 1, 10, 100],
            "clf__solver" : ["lbfgs", "liblinear"],
            "clf__penalty": ["l2"],
        }
    ),
    # 2. SVM con kernel RBF
    (
        "SVM_RBF",
        SVC(kernel="rbf", probability=True, random_state=42),
        {
            "clf__C"    : [0.1, 1, 10, 100],
            "clf__gamma": ["scale", "auto", 0.001, 0.01],
        }
    ),
    # 3. SVM lineal
    (
        "SVM_Linear",
        SVC(kernel="linear", probability=True, random_state=42),
        {"clf__C": [0.01, 0.1, 1, 10]}
    ),
    # 4. K-Nearest Neighbors
    (
        "KNeighbors",
        KNeighborsClassifier(),
        {
            "clf__n_neighbors": [3, 5, 7, 11, 15],
            "clf__weights"    : ["uniform", "distance"],
            "clf__metric"     : ["euclidean", "manhattan"],
        }
    ),
    # 5. Árbol de decisión
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {
            "clf__max_depth"       : [3, 5, 7, 10, None],
            "clf__min_samples_split": [2, 5, 10],
            "clf__criterion"       : ["gini", "entropy"],
        }
    ),
    # 6. Random Forest
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {
            "clf__n_estimators"    : [100, 200, 300],
            "clf__max_depth"       : [5, 10, 20, None],
            "clf__min_samples_split": [2, 5],
            "clf__max_features"    : ["sqrt", "log2"],
        }
    ),
    # 7. Extra Trees
    (
        "ExtraTrees",
        ExtraTreesClassifier(random_state=42, n_jobs=-1),
        {
            "clf__n_estimators"    : [100, 200, 300],
            "clf__max_depth"       : [5, 10, None],
            "clf__min_samples_split": [2, 5],
        }
    ),
    # 8. Gradient Boosting
    (
        "GradientBoosting",
        GradientBoostingClassifier(random_state=42),
        {
            "clf__n_estimators" : [100, 200],
            "clf__learning_rate": [0.05, 0.1, 0.2],
            "clf__max_depth"    : [3, 5, 7],
        }
    ),
    # 9. AdaBoost
    (
        "AdaBoost",
        AdaBoostClassifier(random_state=42, algorithm="SAMME"),
        {
            "clf__n_estimators" : [50, 100, 200],
            "clf__learning_rate": [0.5, 1.0, 2.0],
        }
    ),
    # 10. Red neuronal (MLP)
    (
        "MLP",
        MLPClassifier(max_iter=500, random_state=42),
        {
            "clf__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "clf__alpha"             : [1e-4, 1e-3, 1e-2],
            "clf__activation"        : ["relu", "tanh"],
        }
    ),
    # 11. Naive Bayes
    (
        "NaiveBayes",
        GaussianNB(),
        {"clf__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]}
    ),
    # 12. SGD Classifier
    (
        "SGD",
        SGDClassifier(loss="modified_huber", random_state=42,
                      max_iter=1000),
        {
            "clf__alpha"  : [1e-5, 1e-4, 1e-3, 1e-2],
            "clf__penalty": ["l2", "elasticnet"],
        }
    ),

    (
        "XGBoost",
        XGBClassifier(random_state=42, eval_metric="logloss",
                      use_label_encoder=False, n_jobs=-1),
        {
            "clf__n_estimators" : [100, 200, 300],
            "clf__max_depth"    : [3, 5, 7],
            "clf__learning_rate": [0.05, 0.1, 0.2],
            "clf__subsample"    : [0.8, 1.0],
        }
    ),

    (
        "LightGBM",
        LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
        {
            "clf__n_estimators" : [100, 200, 300],
            "clf__num_leaves"   : [31, 63, 127],
            "clf__learning_rate": [0.05, 0.1, 0.2],
        }
    )
]

Train: 429 muestras | Test: 108 muestras
  Train activos: 292 (68.1%)
  Test  activos: 73 (67.6%)


('LightGBM',
 LGBMClassifier(n_jobs=-1, random_state=42, verbose=-1),
 {'clf__n_estimators': [100, 200, 300],
  'clf__num_leaves': [31, 63, 127],
  'clf__learning_rate': [0.05, 0.1, 0.2]})

In [10]:
def evaluate(model, X_tr, X_te, y_tr, y_te):
    """Devuelve dict con métricas en train y test para análisis de overfitting."""
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
 
    try:
        auc_tr = roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1])
        auc_te = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    except AttributeError:
        auc_tr = auc_te = np.nan
 
    return {
        "acc_train"     : accuracy_score(y_tr, y_pred_tr),
        "acc_test"      : accuracy_score(y_te, y_pred_te),
        "f1_train"      : f1_score(y_tr, y_pred_tr),
        "f1_test"       : f1_score(y_te, y_pred_te),
        "roc_auc_train" : auc_tr,
        "roc_auc_test"  : auc_te,
        "mcc_train"     : matthews_corrcoef(y_tr, y_pred_tr),
        "mcc_test"      : matthews_corrcoef(y_te, y_pred_te),
        "precision_test": precision_score(y_te, y_pred_te),
        "recall_test"   : recall_score(y_te, y_pred_te),
        "overfit_gap"   : accuracy_score(y_tr, y_pred_tr) - accuracy_score(y_te, y_pred_te),
    }

In [11]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
all_results    = []
best_estimators = {}
 
n_total = len(MODEL_DEFINITIONS) * len(feature_sets)
print(f"\nEntrenando {len(MODEL_DEFINITIONS)} modelos × "
      f"{len(feature_sets)} feature sets = {n_total} experimentos...\n")
 
for fs_name, feat_cols in feature_sets.items():
    X_tr = X_train_full[feat_cols]
    X_te = X_test_full[feat_cols]
 
    for model_name, estimator, param_grid in MODEL_DEFINITIONS:
        key = f"{model_name}__{fs_name}"
        print(f"  ▶ {key}")
 
        # Pipeline: imputación → escalado → clasificador
        pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler",  StandardScaler()),
            ("clf",     estimator),
        ])
 
        gs = GridSearchCV(
            pipe, param_grid,
            cv=cv,
            scoring="roc_auc",
            n_jobs=2,
            refit=True,
            return_train_score=True,
        )
        gs.fit(X_tr, y_train)
        best_estimators[key] = gs.best_estimator_
 
        metrics = evaluate(gs.best_estimator_, X_tr, X_te, y_train, y_test)
        metrics["model"]           = model_name
        metrics["feature_set"]     = fs_name
        metrics["n_features"]      = len(feat_cols)
        metrics["best_params"]     = str(gs.best_params_)
        metrics["cv_roc_auc_mean"] = gs.best_score_
        metrics["cv_roc_auc_std"]  = gs.cv_results_["std_test_score"][gs.best_index_]
 
        all_results.append(metrics)
        print(f"      CV AUC={gs.best_score_:.3f} | "
              f"Test AUC={metrics['roc_auc_test']:.3f} | "
              f"Gap={metrics['overfit_gap']:.3f}")
 
 
df_results = pd.DataFrame(all_results).sort_values("roc_auc_test", ascending=False)
df_results.to_csv(f"paso6_resultados_todos.csv", index=False)
print(f"\n✅ paso6_resultados_todos.csv guardado.")


Entrenando 12 modelos × 7 feature sets = 84 experimentos...

  ▶ LogisticRegression__pearson_thr0.7
      CV AUC=0.980 | Test AUC=0.944 | Gap=0.018
  ▶ SVM_RBF__pearson_thr0.7
      CV AUC=0.979 | Test AUC=0.950 | Gap=0.055
  ▶ SVM_Linear__pearson_thr0.7
      CV AUC=0.970 | Test AUC=0.958 | Gap=0.090
  ▶ KNeighbors__pearson_thr0.7


c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


      CV AUC=0.984 | Test AUC=0.942 | Gap=0.065
  ▶ DecisionTree__pearson_thr0.7
      CV AUC=0.957 | Test AUC=0.931 | Gap=0.062
  ▶ RandomForest__pearson_thr0.7
      CV AUC=0.984 | Test AUC=0.953 | Gap=0.056
  ▶ ExtraTrees__pearson_thr0.7
      CV AUC=0.987 | Test AUC=0.953 | Gap=0.060
  ▶ GradientBoosting__pearson_thr0.7
      CV AUC=0.983 | Test AUC=0.963 | Gap=0.065
  ▶ AdaBoost__pearson_thr0.7


c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


      CV AUC=0.985 | Test AUC=0.950 | Gap=0.081
  ▶ MLP__pearson_thr0.7


...

      CV AUC=0.985 | Test AUC=0.940 | Gap=0.065
  ▶ NaiveBayes__spearman_thr0.9
      CV AUC=0.940 | Test AUC=0.907 | Gap=0.083
  ▶ SGD__spearman_thr0.9
      CV AUC=0.980 | Test AUC=0.929 | Gap=0.028
  ▶ LogisticRegression__no_filter
      CV AUC=0.984 | Test AUC=0.955 | Gap=0.032
  ▶ SVM_RBF__no_filter
      CV AUC=0.984 | Test AUC=0.966 | Gap=0.058
  ▶ SVM_Linear__no_filter
      CV AUC=0.979 | Test AUC=0.953 | Gap=0.048
  ▶ KNeighbors__no_filter
      CV AUC=0.982 | Test AUC=0.947 | Gap=0.056
  ▶ DecisionTree__no_filter
      CV AUC=0.959 | Test AUC=0.927 | Gap=0.078
  ▶ RandomForest__no_filter
      CV AUC=0.985 | Test AUC=0.954 | Gap=0.053
  ▶ ExtraTrees__no_filter
      CV AUC=0.987 | Test AUC=0.969 | Gap=0.056
  ▶ GradientBoosting__no_filter
      CV AUC=0.986 | Test AUC=0.972 | Gap=0.056
  ▶ AdaBoost__no_filter


c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


      CV AUC=0.984 | Test AUC=0.967 | Gap=0.056
  ▶ MLP__no_filter


c:\Users\angel\miniconda3\envs\tfm\lib\site-packages\sklearn\model_selection\_search.py:409: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.array(param_list)


      CV AUC=0.985 | Test AUC=0.948 | Gap=0.065
  ▶ NaiveBayes__no_filter
      CV AUC=0.927 | Test AUC=0.900 | Gap=0.085
  ▶ SGD__no_filter
      CV AUC=0.984 | Test AUC=0.901 | Gap=0.051

✅ paso6_resultados_todos.csv guardado.


7. Evaluacion y seleccion del modelo 

In [ ]:
df_results["score_balanced"] = (
    df_results["roc_auc_test"] - 0.5 * df_results["overfit_gap"].abs()
)
df_results = df_results.sort_values("score_balanced", ascending=False)
df_results.to_csv("paso7_resultados_rankeados.csv",sep=";", index=False)
 
top3     = df_results.head(3)
best_row = df_results.iloc[0]
best_key = f"{best_row['model']}__{best_row['feature_set']}"
 
print(f"\nTOP 3 modelos (criterio: ROC-AUC_test − 0.5×|overfit_gap|):\n")
cols_show = ["model", "feature_set", "n_features",
             "roc_auc_test", "f1_test", "mcc_test",
             "overfit_gap", "score_balanced"]
print(top3[cols_show].to_string(index=False))


TOP 3 modelos (criterio: ROC-AUC_test − 0.5×|overfit_gap|):

           model    feature_set  n_features  roc_auc_test  f1_test  mcc_test  overfit_gap  score_balanced
GradientBoosting      no_filter         944      0.972407 0.958904  0.873190     0.055556        0.944629
        AdaBoost pearson_thr0.8         192      0.961840 0.965986  0.893736     0.036972        0.943353
      ExtraTrees      no_filter         944      0.969472 0.959459  0.872078     0.055556        0.941694


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (_, row) in zip(axes, top3.iterrows()):
    k   = f"{row['model']}__{row['feature_set']}"
    est = best_estimators[k]
    fs  = feature_sets[row["feature_set"]]
    cm  = confusion_matrix(y_test, est.predict(X_test_full[fs]))
    ConfusionMatrixDisplay(cm, display_labels=["Inactivo", "Activo"]).plot(
        ax=ax, colorbar=False, cmap="Blues"
    )
    ax.set_title(f"{row['model']}\n{row['feature_set']}\n"
                 f"AUC={row['roc_auc_test']:.3f}", fontsize=9)
plt.tight_layout()
plt.savefig("paso7_confusion_matrices_top3.png",
            dpi=150, bbox_inches="tight")
plt.close()

In [14]:
pivot_auc = df_results.pivot_table(
    index="model", columns="feature_set",
    values="roc_auc_test", aggfunc="max"
)
fig, ax = plt.subplots(figsize=(14, 6))
pivot_auc.plot(kind="bar", ax=ax, edgecolor="white")
ax.axhline(0.8, color="red", linestyle="--", linewidth=1, label="0.8")
ax.set_title("ROC-AUC (Test) por Modelo y Feature Set")
ax.set_ylabel("ROC-AUC")
ax.set_xlabel("")
ax.legend(title="Feature Set", bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("paso7_comparacion_modelos.png",
            dpi=150, bbox_inches="tight")
plt.close()

In [18]:
pivot_gap = df_results.pivot_table(
    index="model", columns="feature_set",
    values="overfit_gap", aggfunc="min"
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot_gap, annot=True, fmt=".2f", cmap="RdYlGn_r",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Overfit Gap (acc_train − acc_test)\nverde = bajo overfitting")
plt.tight_layout()
plt.savefig("paso7_overfit_heatmap.png",
            dpi=150, bbox_inches="tight")
plt.close()

In [19]:
best_est = best_estimators[best_key]
best_fs  = feature_sets[best_row["feature_set"]]
y_pred   = best_est.predict(X_test_full[best_fs])
 
print(f"\n{'='*60}")
print(f"MEJOR MODELO : {best_row['model']}")
print(f"Feature set  : {best_row['feature_set']} ({int(best_row['n_features'])} descriptores)")
print(f"{'='*60}")
print(classification_report(y_test, y_pred, target_names=["Inactivo", "Activo"]))
print(f"MCC = {matthews_corrcoef(y_test, y_pred):.4f}")
try:
    y_prob = best_est.predict_proba(X_test_full[best_fs])[:, 1]
    print(f"ROC-AUC = {roc_auc_score(y_test, y_prob):.4f}")
except AttributeError:
    pass
 
pd.DataFrame(
    classification_report(y_test, y_pred,
                           target_names=["Inactivo", "Activo"],
                           output_dict=True)
).T.to_csv("paso7_report_mejor_modelo.csv")
 
print(f"\n✅ Gráficos y reportes del paso 7 guardados.")


MEJOR MODELO : GradientBoosting
Feature set  : no_filter (944 descriptores)
              precision    recall  f1-score   support

    Inactivo       0.91      0.91      0.91        35
      Activo       0.96      0.96      0.96        73

    accuracy                           0.94       108
   macro avg       0.94      0.94      0.94       108
weighted avg       0.94      0.94      0.94       108

MCC = 0.8732
ROC-AUC = 0.9724

✅ Gráficos y reportes del paso 7 guardados.
